In [12]:
# -*- coding: utf-8 -*-
"""

This notebook performs batch processing on calcium imaging data.
It processes all images in a folder structure, each with its associated ROI set.
"""

import numpy as np
import matplotlib.pyplot as plt
import os
from google.colab import drive
import shutil
from numba import jit
from tifffile import imread
import skimage as ski
import skimage.io as skio
import csv
import seaborn as sns
import pandas as pd
from scipy import stats
import zipfile

# Install required packages
!pip install whittaker_eilers roifile
from whittaker_eilers import WhittakerSmoother

!pip install roifile
import roifile
from roifile import roiread


# Mount Google Drive
drive.mount('/content/drive')

###########################             ###########################
########################### MAIN FUNCTION ###########################
###########################             ###########################

def DFOF_optimized_roi(image_stack, dict_rois_name_xy, opto_list):
    total_frame, total_pixel = image_stack.shape[0:2]
    frame_rate = 50
    smooth_frame = 12 * frame_rate // 2  # smooth over 2 seconds

    # strategy for opto-genetics correction: take an ROI of same size outside of image and average it out of all ROI traces
    # opto_list contains True/false to see if there is opto stimulus, stimulus start time in frame number and stimulus duration in frames
    counter = opto_list[0]
    if counter == True:
      stimulus_start = opto_list[1]
      stimulus_duration = opto_list[2]

    ### initialise dictionaries to stock information
    dict_rois_df_f = dict()
    dict_rois_baseline = dict()
    dict_rois_df = dict()
    dict_opto_df = dict()

    ########## loop over every ROI
    # Create a binary mask that marks your regions of interest
    for roi_name, roi_coordinates in dict_rois_name_xy.items():
      #run the function per key and store result in new dictionary
      print(roi_name)

      ############################ make the ROI a disk mask ############################
      # Compute the center of the circle
      center_x = np.mean(roi_coordinates[:, 0])
      center_y = np.mean(roi_coordinates[:, 1])

      # Compute the radius as the maximum distance from the center to the boundary points
      radius = int(np.max(np.linalg.norm(roi_coordinates - [center_x, center_y], axis=1)))
      # Create a coordinate grid for the entire image
      y_indices, x_indices = np.meshgrid(np.arange(total_pixel), np.arange(total_pixel), indexing="ij")

      # Compute the distance from each pixel to the circle center
      distance_from_center = np.sqrt((x_indices - center_x) ** 2 + (y_indices - center_y) ** 2)

      # Create the binary mask (True for pixels inside the circle, False outside)
      roi_mask = distance_from_center <= radius

      # Get indices of pixels inside the ROI
      roi_indices =(np.where(roi_mask))
      # Unpack x and y coordinates for advanced indexing
      roi_y = roi_indices[0].astype(int)
      roi_x = roi_indices[1].astype(int)

      # Pre-allocate arrays with the same shape as image_stack
      pixel_roi = len(roi_x)

      ############################ Optogenetics artifact correction ############################
      # Find the boundaries of the rectangle enclosing all ROIs
      all_roi_coords = np.concatenate(list(dict_rois_name_xy.values()))
      min_x = int(np.min(all_roi_coords[:, 0]))
      max_x = int(np.max(all_roi_coords[:, 0]))
      min_y = int(np.min(all_roi_coords[:, 1]))
      max_y = int(np.max(all_roi_coords[:, 1]))

      ############# make the opto ROI a disk mask #############
      # Calculate opto ROI center outside the rectangle, at the same y-coordinate
      # Translate the original ROI coordinates
      opto_roi_x = roi_x + (max_x + radius + 10 - int(center_x))  # Shift x-coordinates
      opto_roi_y = roi_y  # Keep y-coordinates the same

      ### Initialize opto_df with the correct shape
      opto_df = np.zeros((total_frame, (pixel_roi)))  # Shape: (total_frames, num_pixels_in_ROI)

      ##### Calculate baseline for each pixel in the opto ROI
      if counter == True:
        opto_baseline = np.mean(image_stack[:stimulus_start - 1, opto_roi_y, opto_roi_x], axis=0)

        ##### Compute the df of opto artifact for each pixel
        for frame in range(stimulus_start, stimulus_start + stimulus_duration):
            opto_df[frame, :] = image_stack[frame, opto_roi_y, opto_roi_x] - (opto_baseline)  # Subtract baseline from each pixel

      dict_opto_df[roi_name] = opto_df  # Store pixel-wise opto_df in the dictionary

      ############################ COMPUTE - DF - BASELINE - DF F ############################
      ### initialise arrays
      df = np.zeros((total_frame,pixel_roi))
      df_f = np.zeros((total_frame,pixel_roi))
      baseline = np.zeros((total_frame,pixel_roi))

      ### First section: frames with increasing window sizes
      for frame in range(smooth_frame):
          # Use vectorized percentile computation over all pixels at once
          window = image_stack[:smooth_frame + frame, roi_y, roi_x]
          baseline[frame] = np.mean(window, axis = 0) ### axis = 0 means in this case along the time series for each individual pixel, not across pixels
          a = image_stack[frame, roi_y, roi_x]
          b = baseline[frame]
          df[frame] = np.where(np.less(image_stack[frame, roi_y, roi_x], baseline[frame]), a - b, image_stack[frame, roi_y, roi_x] - baseline[frame])
          df_f[frame] = df[frame] / baseline[frame]

      # Middle section: frames with full symmetric window
      for frame in range(smooth_frame, total_frame - smooth_frame + 1):
          window = image_stack[frame - smooth_frame:frame + smooth_frame, roi_y, roi_x]
          baseline[frame] = np.mean(window, axis = 0)
          a = image_stack[frame, roi_y, roi_x]
          b = baseline[frame]
          df[frame] = np.where(np.less(image_stack[frame, roi_y, roi_x], baseline[frame]), a - b, image_stack[frame, roi_y, roi_x] - baseline[frame])
          df_f[frame] = df[frame] / baseline[frame]

      # Last section: frames with decreasing window sizes at the end
      for frame in range(total_frame - smooth_frame + 1, total_frame):
          window = image_stack[frame - smooth_frame:total_frame, roi_y, roi_x]
          baseline[frame] = np.mean(window, axis = 0)
          a = image_stack[frame, roi_y, roi_x]
          b = baseline[frame]
          df[frame] = np.where(np.less(image_stack[frame, roi_y, roi_x], baseline[frame]), a - b, image_stack[frame, roi_y, roi_x] - baseline[frame])
          df_f[frame] = df[frame] / baseline[frame]

      # Fix artifact at frame 759 if it exists and if the frame number is in range
      if df_f.shape[0] > 759:
        df_f[759] = df_f[757]  ### artifact at frame 759 corrected by minimal interpolation with previous data point
        df_f[758] = df_f[757]

      dict_rois_df_f[roi_name] = df_f
      dict_rois_baseline[roi_name] = baseline
      dict_rois_df[roi_name] = df
    return dict_rois_baseline, dict_rois_df, dict_rois_df_f, dict_opto_df

def plot_image_overlays(image, overlays, roi_names, save_path=None, **kwargs):
    """Plot image and overlays (bytes) using matplotlib."""
    fig, ax = plt.subplots()
    ax.imshow(image, cmap='plasma')
    if not isinstance(overlays, list):
        overlays = [overlays]

    # Assuming roi_names is a list with the same length as overlays
    for i, overlay in enumerate(overlays):
        roi = overlay
        roi.plot(ax, color='gray', **kwargs)

        # Get ROI center coordinates for text placement
        # Calculate the centroid manually using the coordinates:
        coordinates = roi.coordinates()  # Get coordinates of the ROI
        x_center = np.mean(coordinates[:, 0])  # Calculate mean of x-coordinates
        y_center = np.mean(coordinates[:, 1])  # Calculate mean of y-coordinates

        # Display ROI name as text
        ax.text(x_center, y_center, roi_names[i], color='white')

        ax.axis('off')

    if save_path:
        plt.savefig(save_path)
    plt.close()  # Close the figure to free memory

def process_single_image(image_path, roi_folder_path, output_folder, use_opto=True, stimulus_start=760, stimulus_duration=250):
    """Process a single calcium imaging file with its ROI set"""
    print(f"Processing image: {image_path}")

    # Create specific output folders
    raw_data_folder = os.path.join(output_folder, "Raw Data All Pixels")
    avg_data_folder = os.path.join(output_folder, "Average Data - 1 Pixel per ROI")
    results_folder = os.path.join(output_folder, "Results")

    # Create these folders if they don't exist
    os.makedirs(raw_data_folder, exist_ok=True)
    os.makedirs(avg_data_folder, exist_ok=True)
    os.makedirs(results_folder, exist_ok=True)

    # Load image data
    image_data = ski.io.imread(image_path)
    image_name = os.path.basename(image_path)

    # Set up opto parameters
    opto_list = []
    if use_opto:
        opto_list.append(True)
        opto_list.append(stimulus_start)
        opto_list.append(stimulus_duration)
    else:
        opto_list.append(False)

    # Extract ROIs
    rois = []
    rois_coordinates = []
    dict_rois_name_xy = dict()

    # Get list of ROI files
    roi_files = [f for f in os.listdir(roi_folder_path) if f.endswith('.roi')]

    for roi_file in roi_files:
        # Read ROIs 1 by 1 from ROI folder
        roi_path = os.path.join(roi_folder_path, roi_file)
        roi = roiread(roi_path)
        rois.append(roi)
        rois_coordinates.append(roi.coordinates())

        stripped_roi_file = roi_file.replace('.roi', '')

        # Create dictionary that links name to coordinates
        dict_rois_name_xy[stripped_roi_file] = roi.coordinates()

    # Process data
    dict_rois_baseline, dict_rois_df, dict_rois_df_f, dict_opto_df = DFOF_optimized_roi(image_data, dict_rois_name_xy, opto_list)

    roi_names = list(dict_rois_name_xy.keys())

    # Save ROI overlay image if we have frames
    if image_data.shape[0] > 600:  # Check if we have at least 600 frames
        overlay_path = os.path.join(results_folder, f"ROI_overlay_{image_name.replace('.tif', '')}.png")
        plot_image_overlays(image_data[600], rois, roi_names, save_path=overlay_path)

    # Initialize Whittaker smoother for later use
    whittaker_smoother = WhittakerSmoother(
        lmbda=100, order=1, data_length=(dict_rois_df_f[list(dict_rois_df_f.keys())[0]].shape[0])
    )

    # Generate and save trace plots
    # Plot average pixel trace for each ROI in separate subplots
    num_rois = len(dict_rois_name_xy)
    fig, axes = plt.subplots(num_rois, 1, figsize=(10, 6 * num_rois), sharex=True)

    # Ensure axes is always a list even with a single ROI
    if num_rois == 1:
        axes = [axes]

    # Loop through ROIs and plot on respective subplots
    for i, roi_name in enumerate(dict_rois_name_xy):
        dff = dict_rois_df_f[roi_name]
        opto_df = dict_opto_df[roi_name]
        baseline = dict_rois_baseline[roi_name]

        avg_pixel_df_f = np.mean(dff - opto_df/baseline, axis=1)  # Calculate average dff across all pixels for this ROI

        if use_opto:
            axes[i].vlines(stimulus_start, -1, 1, linestyles='dashed', colors='red')
        axes[i].plot(avg_pixel_df_f)  # Plot the average dff trace
        axes[i].set_ylabel('Intensity')
        axes[i].set_title(f'Average Pixel Trace for ROI {roi_name}')
        axes[i].set_ylim(-1, 1)

    # Set common x-axis label for the entire figure
    plt.xlabel('Time Frame')
    plt.tight_layout()  # Adjust spacing to prevent overlap
    traces_path = os.path.join(results_folder, f"Average_Traces_{image_name.replace('.tif', '')}.png")
    plt.savefig(traces_path)
    plt.close()

    # Plot smoothed traces with left/right overlay
    # Generate plot with offset for left and right ROIs
    fig, ax = plt.subplots(figsize=(10, 6), dpi=600)

    # Custom sorting function to prioritize 'T' ROIs
    def sort_roi_names(roi_name):
        if roi_name.startswith('T'):
            return (0, roi_name)  # 'T' ROIs come first
        else:
            return (1, roi_name)  # Other ROIs come after

    sorted_rois_name_xy = sorted(dict_rois_name_xy, key=sort_roi_names)

    offset_height = 0.35  # Define vertical spacing between traces

    # Loop through ROIs and plot with offsets
    for i, roi_name in enumerate(sorted_rois_name_xy):
        if i % 2 == 0:  # Even index (left side ROIs)
            offset = -offset_height * (i)  # Divide by 2 for proper spacing
            color = 'pink'  # Set color to pink for left side
        else:  # Odd index (right side ROIs)
            offset = -offset_height * ((i - 1))  # Divide by 2 for proper spacing
            color = 'gray'  # Set color to gray for right side

        dff = dict_rois_df_f[roi_name]
        baseline = dict_rois_baseline[roi_name]
        opto_df = dict_opto_df[roi_name]

        avg_pixel_df_f = np.mean(dff - opto_df/baseline, axis=1)  # Calculate average dff across all pixels for this ROI
        smoothed_df_f = np.array(whittaker_smoother.smooth(avg_pixel_df_f))

        # Plot the smoothed trace with offset
        ax.plot(smoothed_df_f + offset, color=color)

        # Add text label next to the trace
        ax.text(len(smoothed_df_f), smoothed_df_f[-1] + offset, roi_name, color=color,
                ha='left', va='center')

        # Add a zero line for the current offset
        ax.plot(np.zeros_like(smoothed_df_f) + offset, color='black', linestyle='--', linewidth=0.5, alpha=0.8, zorder=1)

    ax.set_yticks([])
    # Set labels and legend
    ax.set_xlabel('Time Frame')
    ax.set_ylabel('ΔF/F0')
    ax.legend(loc='upper right')
    plt.title('Smoothed Traces with Vertical Offset and Zero Lines')

    smoothed_path = os.path.join(results_folder, f"Smoothed_Traces_{image_name.replace('.tif', '')}.png")
    plt.savefig(smoothed_path)
    plt.close()

    # Calculate correlation matrices
    # Create dictionary of smoothed average traces
    dict_smooth_avg_pixel_df_f = dict()
    # Loop through ROIs
    for roi_name in dict_rois_df_f:
        dff = dict_rois_df_f[roi_name]
        baseline = dict_rois_baseline[roi_name]
        opto_df = dict_opto_df[roi_name]

        avg_pixel_df_f = np.mean(dff - opto_df/baseline, axis=1)  # Calculate average across all pixels
        smoothed_df_f = np.array(whittaker_smoother.smooth(avg_pixel_df_f))
        dict_smooth_avg_pixel_df_f[roi_name] = smoothed_df_f

    # Custom sorting function to prioritize 'T' ROIs
    sorted_roi_names = sorted(dict_rois_name_xy, key=sort_roi_names)

    # Create an empty correlation matrix
    num_rois = len(sorted_roi_names)
    correlation_matrix = np.zeros((num_rois, num_rois))

    # Calculate correlations between all pairs of ROIs (using sorted roi_names)
    for i in range(num_rois):
        for j in range(i + 1, num_rois):  # Avoid redundant calculations
            roi1_data = dict_smooth_avg_pixel_df_f[sorted_roi_names[i]]
            roi2_data = dict_smooth_avg_pixel_df_f[sorted_roi_names[j]]
            correlation, _ = stats.pearsonr(roi1_data, roi2_data)
            correlation_matrix[i, j] = correlation_matrix[j, i] = correlation

    # Create a DataFrame for the correlation matrix
    df_corr = pd.DataFrame(correlation_matrix, index=sorted_roi_names, columns=sorted_roi_names)

    # Plot the correlation matrix
    plt.figure(figsize=(10, 8), dpi=600)
    sns.heatmap(df_corr, annot=True, cmap='viridis', fmt=".2f", linewidths=.5)
    plt.title('Correlation Matrix of ROIs (Ordered)')
    plt.tight_layout()

    corr_path = os.path.join(results_folder, f"Correlation_Matrix_{image_name.replace('.tif', '')}.png")
    plt.savefig(corr_path)
    plt.close()

    # Plot clustered correlation matrix
    plt.figure(figsize=(10, 8), dpi=600)
    linkage = sns.clustermap(df_corr, method="average", metric="euclidean",
                             row_cluster=True, col_cluster=True, figsize=(10, 8),
                             cmap='viridis', annot=True, fmt=".2f", linewidths=.5)
    plt.title('Clustered Correlation Matrix of ROIs')
    clustered_corr_path = os.path.join(results_folder, f"Clustered_Correlation_Matrix_{image_name.replace('.tif', '')}.png")
    plt.savefig(clustered_corr_path)
    plt.close()

    # If using optogenetics, calculate correlation matrix during stimulation
    if use_opto:
        correlation_matrix_stim = np.zeros((num_rois, num_rois))

        for i in range(num_rois):
            for j in range(i + 1, num_rois):
                roi1_data = dict_smooth_avg_pixel_df_f[sorted_roi_names[i]][stimulus_start:stimulus_start+stimulus_duration]
                roi2_data = dict_smooth_avg_pixel_df_f[sorted_roi_names[j]][stimulus_start:stimulus_start+stimulus_duration]
                correlation, _ = stats.pearsonr(roi1_data, roi2_data)
                correlation_matrix_stim[i, j] = correlation_matrix_stim[j, i] = correlation

        df_corr_stim = pd.DataFrame(correlation_matrix_stim, index=sorted_roi_names, columns=sorted_roi_names)

        plt.figure(figsize=(10, 8), dpi=600)
        sns.heatmap(df_corr_stim, annot=True, cmap='viridis', fmt=".2f", linewidths=.5)
        plt.title('Correlation Matrix of ROIs DURING STIMULATION (Ordered)')
        plt.tight_layout()

        corr_stim_path = os.path.join(results_folder, f"Correlation_Matrix_During_Stim_{image_name.replace('.tif', '')}.png")
        plt.savefig(corr_stim_path)
        plt.close()

        plt.figure(figsize=(10, 8), dpi=600)
        linkage = sns.clustermap(df_corr_stim, method="average", metric="euclidean",
                                 row_cluster=True, col_cluster=True, figsize=(10, 8),
                                 cmap='viridis', annot=True, fmt=".2f", linewidths=.5)
        plt.title('Clustered Correlation Matrix DURING STIMULATION')

        clustered_corr_stim_path = os.path.join(results_folder, f"Clustered_Correlation_Matrix_During_Stim_{image_name.replace('.tif', '')}.png")
        plt.savefig(clustered_corr_stim_path)
        plt.close()

    # Save data as CSV files
    # 1. Save baseline values for all pixels per ROI
    baseline_file = os.path.join(raw_data_folder, f"Baseline_Values_Allpixels_perROIs_{image_name.replace('.tif', '')}.csv")

    with open(baseline_file, "w", newline="") as f:
        header = list(dict_rois_baseline.keys())
        repeated_header = []

        for item in header:
            repeats = (dict_rois_baseline[item].shape)[1]
            repeated_header.extend([item] * repeats)

        w = csv.writer(f)
        w.writerow(repeated_header)

        for i in range(dict_rois_baseline[header[0]].shape[0]):
            row_data = [dict_rois_baseline[roi_name][i,:] for roi_name in header]
            flattened_row_data = [item for sublist in row_data for item in sublist]
            w.writerow(flattened_row_data)

    # 2. Save DF/F0 values for all pixels per ROI
    df_f0_file = os.path.join(raw_data_folder, f"DF_F0_Values_Allpixels_perROIs_{image_name.replace('.tif', '')}.csv")

    with open(df_f0_file, "w", newline="") as f:
        header = list(dict_rois_df_f.keys())
        repeated_header = []

        for item in header:
            repeats = (dict_rois_df_f[item].shape)[1]
            repeated_header.extend([item] * repeats)

        w = csv.writer(f)
        w.writerow(repeated_header)

        for i in range(dict_rois_df_f[header[0]].shape[0]):
            row_data = [dict_rois_df_f[roi_name][i,:] for roi_name in header]
            flattened_row_data = [item for sublist in row_data for item in sublist]
            w.writerow(flattened_row_data)

    # 3. Save average DF/F0 (1 value per ROI)
    dict_avg_pixel_df_f = dict()
    for roi_name in dict_rois_df_f:
        dff = dict_rois_df_f[roi_name]
        opto_df = dict_opto_df[roi_name]
        baseline = dict_rois_baseline[roi_name]
        dict_avg_pixel_df_f[roi_name] = np.mean(dff - opto_df/baseline, axis=1)

    avg_df_f0_file = os.path.join(avg_data_folder, f"Average_DF_F0_Values_1pixel_perROIs_{image_name.replace('.tif', '')}.csv")

    with open(avg_df_f0_file, "w", newline="") as f:
        header = list(dict_rois_baseline.keys())
        w = csv.writer(f)
        w.writerow(header)

        for i in range(dict_avg_pixel_df_f[header[0]].shape[0]):
            row_data = [dict_avg_pixel_df_f[roi_name][i] for roi_name in header]
            w.writerow(row_data)

    # 4. Save smoothed average DF/F0
    smooth_avg_df_f0_file = os.path.join(avg_data_folder, f"Smooth_Average_DF_F0_Values_1pixel_perROIs_{image_name.replace('.tif', '')}.csv")

    with open(smooth_avg_df_f0_file, "w", newline="") as f:
        header = list(dict_rois_baseline.keys())
        w = csv.writer(f)
        w.writerow(header)

        for i in range(dict_smooth_avg_pixel_df_f[header[0]].shape[0]):
            row_data = [dict_smooth_avg_pixel_df_f[roi_name][i] for roi_name in header]
            w.writerow(row_data)

    print(f"Processing complete for {image_name}")
    return {
        "baseline_file": baseline_file,
        "df_f0_file": df_f0_file,
        "avg_df_f0_file": avg_df_f0_file,
        "smooth_avg_df_f0_file": smooth_avg_df_f0_file,
        "overlay_path": overlay_path,
        "traces_path": traces_path,
        "smoothed_path": smoothed_path,
        "corr_path": corr_path,
        "clustered_corr_path": clustered_corr_path
    }

def extract_roi_set(zip_path, extract_folder):
    """Extract ROI set from zip file to specified folder"""
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_folder)
    return extract_folder

def batch_process_folder(main_folder_path, use_opto=True, stimulus_start=760, stimulus_duration=250):
    """
    Process all images in a folder structure

    Folder structure:
    Main Folder/
    ├── Sub Folder 1/
    │   ├── Image 1/
    │   │   ├── image1.tif
    │   │   └── RoiSet.zip
    │   └── Image 2/
    │       ├── image2.tif
    │       └── RoiSet.zip
    └── Sub Folder 2/
        └── ...

    Returns a summary of processed files
    """
    processed_files = []

    # Get all subdirectories in the main folder
    sub_folders = [f.path for f in os.scandir(main_folder_path) if f.is_dir()]

    for sub_folder in sub_folders:
        print(f"Processing sub-folder: {os.path.basename(sub_folder)}")

        # Get all image folders in this subfolder
        image_folders = [f.path for f in os.scandir(sub_folder) if f.is_dir()]

        for image_folder in image_folders:
            print(f"  Processing image folder: {os.path.basename(image_folder)}")

            # Find the TIF image in this folder
            tif_files = [f.path for f in os.scandir(image_folder) if f.path.lower().endswith('.tif')]

            # Find the ROI set zip
            roi_zip = None
            for f in os.scandir(image_folder):
                if f.name.lower() == 'roiset.zip':
                    roi_zip = f.path
                    break

            if not tif_files or not roi_zip:
                print(f"    Missing TIF image or ROI set in {image_folder}, skipping...")
                continue

            # Use the first TIF file found
            image_path = tif_files[0]

            # Create temporary folder for extracted ROIs
            roi_extract_folder = os.path.join(image_folder, 'ROI_temp')
            os.makedirs(roi_extract_folder, exist_ok=True)

            # Extract ROIs
            extract_roi_set(roi_zip, roi_extract_folder)

            try:
                # Process the image
                result = process_single_image(
                    image_path,
                    roi_extract_folder,
                    image_folder,
                    use_opto=use_opto,
                    stimulus_start=stimulus_start,
                    stimulus_duration=stimulus_duration
                )

                # Add to processed files
                processed_files.append({
                    'sub_folder': os.path.basename(sub_folder),
                    'image_folder': os.path.basename(image_folder),
                    'image_file': os.path.basename(image_path),
                    'output_files': result
                })

                print(f"    Successfully processed {os.path.basename(image_path)}")
            except Exception as e:
                print(f"    Error processing {os.path.basename(image_path)}: {str(e)}")

            # Clean up temporary ROI folder
            shutil.rmtree(roi_extract_folder)

    return processed_files

# Main execution
if __name__ == "__main__":
    # Ask user for main folder path from Google Drive
    main_folder = input("Enter the path to your main folder in Google Drive (e.g., /content/drive/MyDrive/Calcium_Imaging_Data): ")


    # Ask about optogenetics parameters
    use_opto_input = input("Is there optogenetic stimulation? (yes/no, default: yes): ").lower()
    use_opto = False if use_opto_input == "no" else True

    if use_opto:
        stim_start = input("Enter stimulus start frame (default: 760): ")
        stim_start = int(stim_start) if stim_start.isdigit() else 760

        stim_duration = input("Enter stimulus duration in frames (default: 250): ")
        stim_duration = int(stim_duration) if stim_duration.isdigit() else 250
    else:
        stim_start = 760  # Default values even if not used
        stim_duration = 250

    # Print processing information
    print(f"\nStarting batch processing with the following parameters:")
    print(f"Main folder: {main_folder}")
    print(f"Using optogenetics: {use_opto}")
    if use_opto:
        print(f"Stimulus start frame: {stim_start}")
        print(f"Stimulus duration: {stim_duration}")
    print("\n")

    # Process all files in the folder structure
    processed_files = batch_process_folder(
        main_folder_path=main_folder,
        use_opto=use_opto,
        stimulus_start=stim_start,
        stimulus_duration=stim_duration
    )

    # Generate summary
    print("\n=== Processing Summary ===")
    print(f"Total images processed: {len(processed_files)}")

    # Create summary dataframe
    summary_data = []
    for item in processed_files:
        summary_data.append({
            'Sub Folder': item['sub_folder'],
            'Image Folder': item['image_folder'],
            'Image File': item['image_file'],
        })

    summary_df = pd.DataFrame(summary_data)

    # Save summary as CSV
    summary_path = os.path.join(main_folder, "processing_summary.csv")
    summary_df.to_csv(summary_path, index=False)
    print(f"Summary saved to: {summary_path}")

    print("\nBatch processing complete!")
    print("Data and figures have been saved in their respective image folders.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Enter the path to your main folder in Google Drive (e.g., /content/drive/MyDrive/Calcium_Imaging_Data): /content/drive/MyDrive/aaa
Is there optogenetic stimulation? (yes/no, default: yes): 
Enter stimulus start frame (default: 760): 
Enter stimulus duration in frames (default: 250): 

Starting batch processing with the following parameters:
Main folder: /content/drive/MyDrive/aaa
Using optogenetics: True
Stimulus start frame: 760
Stimulus duration: 250


Processing sub-folder: backward wave
  Processing image folder: Image 2
Processing image: /content/drive/MyDrive/aaa/backward wave/Image 2/NOOPTO~1.TIF
A4l
A3l
A2l
A5l
A1l
A7l
A6l
A9r
A8r
A7r
A6r
A5r
A4r
A3r
A2r
A1r
A8l
A9l


<ipython-input-12-a7da0b1fa92e>:324: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(loc='upper right')


Processing complete for NOOPTO~1.TIF
    Successfully processed NOOPTO~1.TIF
  Processing image folder: Image 1
Processing image: /content/drive/MyDrive/aaa/backward wave/Image 1/Fast_Crawl_maybe_20230704_182@11A07-LexA_502@LexAoP-CsChr-UAS-GCamp-OK371_retinal_Number1.lif - 1000Hz b 20V ch duty 50% trial 1_register.TIF
    Error processing Fast_Crawl_maybe_20230704_182@11A07-LexA_502@LexAoP-CsChr-UAS-GCamp-OK371_retinal_Number1.lif - 1000Hz b 20V ch duty 50% trial 1_register.TIF: list index out of range
  Processing image folder: Image 3
Processing image: /content/drive/MyDrive/aaa/backward wave/Image 3/NOOPTO~1.TIF
A4l
A3l
A2l
A5l
A1l
A7l
A6l
A9r
A8r
A7r
A6r
A5r
A4r
A3r
A2r
A1r
A8l
A9l


<ipython-input-12-a7da0b1fa92e>:324: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(loc='upper right')


Processing complete for NOOPTO~1.TIF
    Successfully processed NOOPTO~1.TIF

=== Processing Summary ===
Total images processed: 2
Summary saved to: /content/drive/MyDrive/aaa/processing_summary.csv

Batch processing complete!
Data and figures have been saved in their respective image folders.


<Figure size 6000x4800 with 0 Axes>

<Figure size 6000x4800 with 0 Axes>

<Figure size 6000x4800 with 0 Axes>

<Figure size 6000x4800 with 0 Axes>